# 分布式B+树结构动态更新与局部数据迁移

分布式B+树把全局有序键空间划分为多个连续范围，每个范围由一个物理节点负责，并在节点内部建立局部B+树索引。查询首先通过全局目录确定目标节点，再进入该节点的局部索引；节点加入或删除时，只调整相邻范围并迁移局部记录，从而避免重建整个分布式索引。

本实验基于C++17和CMake实现“全局范围目录+多节点局部B+树”的单机模拟。实验生成全局有序记录，完成点查询、跨节点范围查询、节点加入时的分区拆分和节点删除时的相邻分区合并，并通过负载统计、迁移比例、全局重新均分基线和正确性检查评价动态维护成本。

本节学习大纲如下：

1. 实验概述：介绍实验目标、前置知识和实验要点；
2. 环境准备：创建工程目录并检查C++17/CMake工具链；
3. 问题分析：分析范围分片、两层索引、查询流程、节点增删和迁移指标；
4. 核心程序开发：实现局部B+树、分布式模拟器、指标统计和实验主流程；
5. 结果验证与性能分析：编译运行工程，验证目录、记录和查询正确性并分析迁移成本；
6. 实验总结：归纳分布式范围索引的组织方式和局部数据迁移特点。


---
## 1. 实验概述

本实验以分布式有序索引的动态维护为问题背景，将全局记录按键范围近似均分到多个节点。每个节点保存一个连续键区间、该区间内的记录和一棵局部B+树；全局目录保存“键范围到节点”的映射，用于在查询前完成第一层路由。

实验先验证初始状态下的点查询和范围查询，再模拟新节点加入与已有节点删除。节点加入时从当前最重节点分出后一半连续记录；节点删除时把被删节点记录合并到负载较小的相邻节点。程序同时计算局部迁移量和全局重新均分基线，用于比较不同维护策略的代价。


### 1.1 实验目标

完成本实验后应达到以下目标：

1. 理解分布式B+树中全局目录、范围分片、局部B+树索引和记录存储之间的组织关系，能够说明目录项、物理节点和叶页之间的映射关系。
2. 掌握基于目录路由的点查询和跨节点范围查询流程，能够使用页高键定位局部叶页，并沿叶链完成连续范围扫描。
3. 掌握节点加入时的局部分区拆分和节点删除时的相邻分区合并过程，理解目录边界、节点记录和局部索引需要同步更新的原因。
4. 具备根据正确性检查、迁移数量、迁移比例、全局重分布基线和节点负载变化分析分布式索引维护成本的能力。


### 1.2 前置知识

本实验要求提前具备以下基础：

1. B+树基础：理解内部索引、叶页、有序键和叶页链的作用；
2. 分布式分片基础：理解连续范围分片、节点路由和数据迁移的基本概念；
3. C++与CMake基础：能够阅读C++17工程，了解头文件、源文件和构建脚本之间的关系；
4. 排序与二分查找基础：理解有序数组、`lower_bound`和闭区间相交判断；
5. 统计指标基础：理解平均值、标准差、不均衡率和迁移比例的含义。


### 1.3 实验要点

实验中应重点关注以下内容：

1. 两层索引：全局目录负责节点路由，局部B+树负责节点内记录定位；
2. 连续范围：每个节点持有一个连续且不重叠的键区间，目录顺序必须与键顺序一致；
3. 范围查询：先筛选与查询区间相交的目录项，再合并各节点局部查询结果；
4. 动态维护：节点增删后需要同步更新记录、范围边界、目录项和局部B+树；
5. 结果分析：同时检查正确性、访问节点数、访问叶页数、节点负载和迁移比例。


---
## 2. 环境准备

### 2.1 创建实验目录并检查编译环境

本实验只依赖C++17标准库和CMake，不依赖CANN、HCCL或NPU。Notebook会在`src/03.03_extra_distributed_bplus_tree`目录下生成一份可独立编译运行的工程。

目录划分如下：

- `include/`：保存记录、目录、局部B+树、模拟器、指标和实验接口；
- `src/`：保存局部索引、分布式模拟器、指标统计、实验流程和主程序；
- `scripts/`：保存自动配置、编译和运行脚本；
- `results/`：保存Notebook运行输出。


In [ ]:
from pathlib import Path
import os
import subprocess

WORK_DIR = Path("src/03.03_extra_distributed_bplus_tree").resolve()
for directory in [WORK_DIR / "include", WORK_DIR / "src", WORK_DIR / "scripts", WORK_DIR / "results"]:
    directory.mkdir(parents=True, exist_ok=True)

print("Experiment directory:", WORK_DIR)
print("Current platform:", os.name)
subprocess.run(["c++", "--version"], check=False)
subprocess.run(["cmake", "--version"], check=False)


### 2.2 写入工程公共接口

本节写入记录、全局目录、局部B+树、分布式模拟器、指标和实验流程的公共头文件。实现文件将在第4节写入，CMake构建文件和运行脚本将在第5节写入。运行到第5节后，Notebook生成的目录将具备完整的C++17工程结构。


In [ ]:
%%writefile src/03.03_extra_distributed_bplus_tree/include/record.hpp
#pragma once

#include <cstdint>
#include <string>

namespace dbpt {

    // 实验中的键值记录，key 用于范围分片和 B+ 树索引，value 保存业务数据。
    struct Record {
        std::int64_t key = 0;
        std::string value;
    };

    // 控制记录规模、初始节点、局部叶页容量以及节点增删场景的实验参数。
    struct ExperimentConfig {
        std::size_t record_count = 10000;
        std::size_t initial_nodes = 4;
        std::size_t leaf_capacity = 32; // 每个局部 B+ 树叶页最多容纳的记录数。
        std::string key_distribution = "uniform"; // 支持 uniform、clustered 和 skewed 三种键分布。
        std::string add_node_id = "node_4";
        std::string remove_node_id = "node_1";
        unsigned seed = 2026;
    };

}  // namespace dbpt


In [ ]:
%%writefile src/03.03_extra_distributed_bplus_tree/include/directory.hpp
#pragma once

#include <cstdint>
#include <string>

namespace dbpt {

// 全局目录项：把一个闭区间 [start_key, end_key] 路由到指定物理节点。
struct DirectoryEntry {
    std::int64_t start_key = 0;
    std::int64_t end_key = 0;
    std::string node_id;

    // 判断单个键是否落在当前目录项负责的闭区间内。
    bool contains(std::int64_t key) const;
    // 判断当前目录区间是否与查询区间 [low, high] 相交。
    bool overlaps(std::int64_t low, std::int64_t high) const;
};

}  // namespace dbpt


In [ ]:
%%writefile src/03.03_extra_distributed_bplus_tree/include/local_bplus_index.hpp
#pragma once

#include "record.hpp"

#include <optional>
#include <string>
#include <utility>
#include <vector>

namespace dbpt {

// 简化 B+ 树的叶页：键和值按相同下标对应，next_page_id 形成有序叶链。
struct LeafPage {
    std::vector<std::int64_t> keys;
    std::vector<std::string> values;
    int next_page_id = -1;
};

// 局部范围查询结果，同时记录为完成查询访问的叶页数量。
struct RangeSearchResult {
    std::vector<Record> records;
    std::size_t leaf_pages_touched = 0;
};

// 单节点上的教学化 B+ 树索引：使用页高键数组定位叶页，再在有序叶页中查询。
class LocalBPlusIndex {
public:
    explicit LocalBPlusIndex(std::size_t leaf_capacity = 32);

    void bulk_load(const std::vector<Record>& records); // 将已有记录排序后批量装载为多个叶页，并建立叶页间的 next 指针。
    std::pair<std::optional<std::string>, std::size_t> search(std::int64_t key) const; // 返回点查询结果和访问的叶页数量。
    RangeSearchResult range_search(std::int64_t low, std::int64_t high) const; // 定位起始叶页后沿叶链扫描 [low, high]。

    std::size_t page_count() const;

private:
    std::size_t leaf_capacity_;
    std::vector<LeafPage> pages_;
    std::vector<std::int64_t> page_high_keys_; // 每个叶页的最大键，充当简化的上层目录以支持二分定位。
};

}  // namespace dbpt


In [ ]:
%%writefile src/03.03_extra_distributed_bplus_tree/include/simulator.hpp
#pragma once

#include "directory.hpp"
#include "local_bplus_index.hpp"
#include "record.hpp"

#include <map>
#include <optional>
#include <string>
#include <tuple>
#include <vector>

namespace dbpt {

// 一个分布式节点持有连续键区间、实际记录以及该区间的局部 B+ 树索引。
struct DistributedNode {
    std::string node_id;
    std::int64_t start_key = 0;
    std::int64_t end_key = 0;
    std::vector<Record> records;
    LocalBPlusIndex local_index;

    std::size_t load() const;
};

// 一条点查询或范围查询的正确性与访问代价记录。
struct QueryResult {
    std::string query;
    bool correct = false;
    std::size_t result_count = 0;
    std::size_t touched_nodes = 0;
    std::size_t leaf_pages_touched = 0;
};

// 节点加入或删除时的局部数据迁移结果。
struct MigrationResult {
    std::string event;
    std::string source_node;
    std::string target_node;
    std::size_t migrated_records = 0;
    double migration_ratio = 0.0;
};

// 跨节点范围查询汇总结果。
struct DistributedRangeResult {
    std::vector<Record> records;
    std::size_t touched_nodes = 0;
    std::size_t leaf_pages_touched = 0;
};

// 在单进程内模拟“全局范围目录 + 多节点局部 B+ 树”的分布式索引。
class DistributedBPlusTreeSimulator {
public:
    DistributedBPlusTreeSimulator(std::vector<Record> records,
                                  std::size_t node_count,
                                  std::size_t leaf_capacity);

    const std::vector<DirectoryEntry>& directory() const;
    const DistributedNode& node(const std::string& node_id) const;

    // 通过全局目录定位负责给定键的节点区间。
    std::optional<DirectoryEntry> locate_entry(std::int64_t key) const;
    // 点查询：先目录路由，再进入目标节点的局部 B+ 树。
    std::tuple<std::optional<std::string>, std::optional<std::string>, std::size_t> search(std::int64_t key) const;
    // 范围查询：访问所有与查询区间相交的节点并合并局部结果。
    DistributedRangeResult range_search(std::int64_t low, std::int64_t high) const;

    // 新节点从当前最重节点分走后一半连续记录。
    MigrationResult add_node(const std::string& new_node_id);
    // 删除节点时将其记录迁移到负载较小的相邻节点。
    MigrationResult remove_node(const std::string& node_id);

    std::map<std::string, std::size_t> node_loads() const;
    std::map<std::string, std::size_t> leaf_pages_per_node() const;
    std::map<std::int64_t, std::string> assignment() const;

    std::pair<bool, std::string> validate_directory() const;
    std::pair<bool, std::string> validate_records() const;

private:
    std::vector<Record> records_; // records_ 保存全局参考记录；nodes_ 和 directory_ 保存当前分布状态。
    std::size_t leaf_capacity_; // 局部 B+ 树每个叶页的容量
    std::map<std::string, DistributedNode> nodes_; // node_id -> DistributedNode
    std::vector<DirectoryEntry> directory_; // 键范围 -> node_id

    LocalBPlusIndex build_index(const std::vector<Record>& records) const;
    DistributedNode build_node(const std::string& node_id,
                               std::int64_t start_key,
                               std::int64_t end_key,
                               const std::vector<Record>& records) const;
    void build_initial_tree(std::size_t node_count);
    void sort_directory();
    // 提取有序目录项的结束键，供 locate_entry 使用 lower_bound 二分查找。
    std::vector<std::int64_t> end_keys() const;
};

}  // namespace dbpt


In [ ]:
%%writefile src/03.03_extra_distributed_bplus_tree/include/metrics.hpp
#pragma once

#include "record.hpp"

#include <cstdint>
#include <map>
#include <string>
#include <vector>

namespace dbpt {

// 各节点记录数的统计摘要，用于观察节点增删前后的负载变化。
struct LoadStats {
    std::string label;
    std::map<std::string, std::size_t> node_loads;
    std::size_t min_load = 0;
    std::size_t max_load = 0;
    double avg_load = 0.0;
    double std_load = 0.0;
    // 最大负载 / 平均负载，越接近 1 表示负载越均衡。
    double imbalance_ratio = 0.0;
};

// 根据 node_id -> record_count 计算最小、最大、平均、标准差等指标。
LoadStats compute_load_stats(const std::string& label, const std::map<std::string, std::size_t>& loads);

// 模拟节点数变化后重新均分全部记录，用作“全局重分布”迁移开销基线。
std::map<std::int64_t, std::string> global_redistribution_assignment(const std::vector<Record>& records,
                                                                     std::size_t node_count);

// 比较同一批键在变更前后的节点归属，统计发生迁移的记录数。
std::size_t count_assignment_changes(const std::map<std::int64_t, std::string>& before,
                                     const std::map<std::int64_t, std::string>& after);

}  // namespace dbpt


In [ ]:
%%writefile src/03.03_extra_distributed_bplus_tree/include/experiment.hpp
#pragma once

#include "metrics.hpp"
#include "record.hpp"
#include "simulator.hpp"

#include <vector>

namespace dbpt {

// 按指定分布生成全局有序记录集。
std::vector<Record> generate_records(const std::string& mode, std::size_t record_count, unsigned seed);
// 将有序记录按数量近似均分为连续分片。
std::vector<std::vector<Record>> split_evenly(const std::vector<Record>& records, std::size_t parts);
// 使用串行参考结果验证点查询和范围查询。
std::vector<QueryResult> validate_queries(const DistributedBPlusTreeSimulator& simulator,
                                          const std::vector<Record>& records);

// 执行初始化、查询验证、节点加入、节点删除、指标统计和正确性检查。
void run_experiment(const ExperimentConfig& config);

}  // namespace dbpt


### 2.3 工程公共接口检查

公共头文件写入完成后，检查关键文件是否已经生成。检查结果均为`OK`时，说明实验记录、目录、局部索引、模拟器和指标接口已经就绪。


In [ ]:
required_headers = [
    "include/record.hpp",
    "include/directory.hpp",
    "include/local_bplus_index.hpp",
    "include/simulator.hpp",
    "include/metrics.hpp",
    "include/experiment.hpp",
]

for relative_path in required_headers:
    path = WORK_DIR / relative_path
    print(f"{relative_path}: {'OK' if path.exists() else 'MISSING'}")


---
## 3. 问题分析

本节分析全局记录与连续范围分片、全局目录与局部B+树、点查询和范围查询、节点动态变化以及实验评价指标。后续C++实现将围绕这些数据结构与流程展开。


### 3.1 全局记录与连续范围分片

一条实验记录由整数键`key`和字符串值`value`组成。所有记录按键升序排列后，按记录数量近似均分为$n$个连续分片，第$i$个分片形成闭区间：

$$
R_i=[startKey_i,endKey_i]
$$

每个分片由一个物理节点负责。相邻分片在键空间中保持顺序且不重叠，因此任意已存储键最多只会落入一个节点区间。`uniform`、`clustered`和`skewed`三种键分布会改变键值间距，但初始分片都按记录数量切分，便于观察节点负载和范围边界。


### 3.2 全局目录与局部B+树

全局目录项保存`[start_key,end_key] -> node_id`映射；物理节点保存同一区间的记录和局部B+树。一次查询分成两个阶段：

1. 全局目录根据键或查询区间选择目标节点；
2. 目标节点的局部B+树完成节点内记录查找。

这种两层结构使全局层只维护较小的范围目录，而具体记录索引分散在各节点。节点变化时，只需更新受影响的少量目录项和局部索引。


### 3.3 局部B+树叶页与页高键

教学实现只保留B+树查询所需的叶层结构。`bulk_load`先将节点记录按键排序，再按`leaf_capacity`切分为多个叶页。每个叶页保存平行的`keys`和`values`数组，并通过`next_page_id`连接下一个叶页。

程序额外保存每个叶页的最大键：

$$
highKey_j=\max(keys_j)
$$

`page_high_keys_`相当于简化的上层目录。点查询和范围查询都先使用`lower_bound`找到第一个满足$highKey_j\ge k$的候选叶页，再在叶页内部查找或沿叶链扫描。


### 3.4 点查询流程

对目标键$k$，全局目录先在有序结束键数组中查找：

$$
i=\min\{j\mid endKey_j\ge k\}
$$

找到候选目录项后，还要检查$startKey_i\le k\le endKey_i$，避免把目录间隙中的不存在键错误路由到节点。确认目标节点后，局部索引再次通过页高键二分定位叶页，并在页内二分查找键。程序返回查询值、节点ID和访问叶页数，用于同时验证结果与访问代价。


### 3.5 范围查询流程

对查询区间$Q=[low,high]$，目录项$R_i=[startKey_i,endKey_i]$与查询相交的条件为：

$$
endKey_i\ge low\quad\text{且}\quad startKey_i\le high
$$

程序按目录顺序访问所有相交节点。每个节点先定位可能包含`low`的叶页，再沿叶链扫描，遇到键大于`high`时停止。最后合并所有局部结果并按键排序。查询输出中的`touched_nodes`和`leaf_pages_touched`分别表示跨节点路由范围和局部扫描代价。


### 3.6 节点加入与局部分区拆分

新节点加入时，程序选择当前记录数最多的节点作为源节点，将其有序记录的后一半迁移到新节点。源节点和新节点各自获得连续范围，并重新构建局部B+树；全局目录用两个新目录项替换原目录项。

若源节点负载为$L$，本次局部迁移量约为$\lfloor L/2\rfloor$。这种处理只影响一个原分区及其新相邻分区，不要求其他节点搬迁数据。


### 3.7 节点删除与相邻分区合并

删除节点时，为保持每个节点负责连续范围，不能把记录迁移到任意节点。程序在被删节点的左右相邻节点中选择当前负载较小者作为目标节点，把全部记录合并、排序并重新构建目标节点的局部B+树，同时删除旧目录项并扩大目标区间。

删除事件的迁移量等于被删节点记录数。迁移目标选择较轻相邻节点可以兼顾范围连续性与负载控制，但局部策略不保证变更后达到全局最优均衡。


### 3.8 负载、迁移与正确性指标

设节点数为$n$、总记录数为$N$、节点$i$的负载为$L_i$。理论平均负载、总体标准差和不均衡率分别为：

$$
averageLoad=\frac{N}{n}
$$

$$
stdLoad=\sqrt{\frac{1}{n}\sum_{i=1}^{n}(L_i-averageLoad)^2}
$$

$$
imbalanceRatio=\frac{\max_i L_i}{averageLoad}
$$

节点变更的迁移比例为：

$$
migrationRatio=\frac{migratedRecords}{totalRecords}
$$

程序还模拟节点加入后对全部记录重新均分，计算全局重分布迁移量作为对照基线。正确性检查要求目录有序且不重叠、所有原始记录恰好出现一次，以及点查询和范围查询结果与串行参考结果一致。


### 3.9 实验参数设置

默认实验生成10000条记录，初始节点数为4，局部叶页容量为32，键分布为`uniform`，新增节点为`node_4`，删除节点为`node_1`，随机种子为2026。下面计算节点增删前后的理论平均负载和理想叶页数量。


In [ ]:
RECORD_COUNT = 10000
INITIAL_NODES = 4
LEAF_CAPACITY = 32
KEY_DISTRIBUTION = "uniform"
ADD_NODE_ID = "node_4"
REMOVE_NODE_ID = "node_1"
SEED = 2026

print("recordCount:", RECORD_COUNT)
print("initialNodes:", INITIAL_NODES)
print("leafCapacity:", LEAF_CAPACITY)
print("keyDistribution:", KEY_DISTRIBUTION)
print("average load initially:", RECORD_COUNT / INITIAL_NODES)
print("average load after add:", RECORD_COUNT / (INITIAL_NODES + 1))
print("average load after remove:", RECORD_COUNT / INITIAL_NODES)
print("ideal leaf pages per initial node:", (RECORD_COUNT // INITIAL_NODES + LEAF_CAPACITY - 1) // LEAF_CAPACITY)


理论平均负载只反映全局均分目标。局部拆分和相邻合并优先限制迁移范围，因此节点加入或删除后的实际负载可能偏离理论平均值。实验应把这种偏差与迁移量降低结合起来分析，不能只以单次负载不均衡判断维护策略优劣。


---
## 4. 核心程序开发

本节依次实现局部B+树、分布式模拟器、负载与迁移指标、实验场景和主程序。所有实现均与交付版C++工程保持一致。


### 4.1 局部B+树批量装载与查询

`LocalBPlusIndex`按叶页容量批量装载有序记录，建立页高键数组和叶页链。点查询通过两次二分完成页定位和页内查找；范围查询定位起始页后沿`next_page_id`扫描，并累计访问叶页数。


In [ ]:
%%writefile src/03.03_extra_distributed_bplus_tree/src/local_bplus_index.cpp
#include "local_bplus_index.hpp"

#include <algorithm>
#include <stdexcept>

namespace dbpt {

LocalBPlusIndex::LocalBPlusIndex(std::size_t leaf_capacity)
    : leaf_capacity_(leaf_capacity)
{
    // 叶页容量决定批量装载时每页最多保存多少条记录。
    if (leaf_capacity_ == 0) {
        throw std::invalid_argument("leaf_capacity must be positive");
    }
}

void LocalBPlusIndex::bulk_load(const std::vector<Record>& records)
{
    // B+ 树叶层必须按键有序，先复制排序以免改变调用方数据。
    std::vector<Record> sorted_records = records;
    std::sort(sorted_records.begin(), sorted_records.end(), [](const Record& lhs, const Record& rhs) {
        return lhs.key < rhs.key;
    });

    pages_.clear();
    page_high_keys_.clear();

    // 按 leaf_capacity_ 顺序切页，键和值使用相同下标一一对应。
    for (std::size_t start = 0; start < sorted_records.size(); start += leaf_capacity_) {
        const std::size_t end = std::min(start + leaf_capacity_, sorted_records.size());
        LeafPage page;
        page.keys.reserve(end - start);
        page.values.reserve(end - start);
        for (std::size_t index = start; index < end; ++index) {
            page.keys.push_back(sorted_records[index].key);
            page.values.push_back(sorted_records[index].value);
        }
        pages_.push_back(std::move(page));
    }

    // 建立叶页链，并记录每页最大键作为简化的上层索引。
    for (std::size_t index = 0; index < pages_.size(); ++index) {
        pages_[index].next_page_id = (index + 1 < pages_.size()) ? static_cast<int>(index + 1) : -1;
        page_high_keys_.push_back(pages_[index].keys.back());
    }
}

std::pair<std::optional<std::string>, std::size_t> LocalBPlusIndex::search(std::int64_t key) const
{
    if (pages_.empty()) {
        return {std::nullopt, 0};
    }

    // 第一次二分：定位第一个最大键不小于目标键的候选叶页。
    const auto page_it = std::lower_bound(page_high_keys_.begin(), page_high_keys_.end(), key);
    if (page_it == page_high_keys_.end()) {
        return {std::nullopt, 1};
    }

    const std::size_t page_index = static_cast<std::size_t>(std::distance(page_high_keys_.begin(), page_it));
    const LeafPage& page = pages_[page_index];
    // 第二次二分：在候选叶页内部查找目标键。
    const auto key_it = std::lower_bound(page.keys.begin(), page.keys.end(), key);
    if (key_it != page.keys.end() && *key_it == key) {
        const std::size_t pos = static_cast<std::size_t>(std::distance(page.keys.begin(), key_it));
        return {page.values[pos], 1};
    }
    return {std::nullopt, 1};
}

RangeSearchResult LocalBPlusIndex::range_search(std::int64_t low, std::int64_t high) const
{
    RangeSearchResult result;
    if (low > high || pages_.empty()) {
        return result;
    }

    // 先定位可能包含 low 的第一个叶页，再沿 next_page_id 顺序扫描。
    const auto page_it = std::lower_bound(page_high_keys_.begin(), page_high_keys_.end(), low);
    if (page_it == page_high_keys_.end()) {
        result.leaf_pages_touched = 1;
        return result;
    }

    int current = static_cast<int>(std::distance(page_high_keys_.begin(), page_it));
    while (current >= 0 && static_cast<std::size_t>(current) < pages_.size()) {
        const LeafPage& page = pages_[static_cast<std::size_t>(current)];
        ++result.leaf_pages_touched;
        for (std::size_t index = 0; index < page.keys.size(); ++index) {
            const std::int64_t key = page.keys[index];
            if (key < low) {
                continue;
            }
            if (key > high) {
                return result;
            }
            result.records.push_back(Record{key, page.values[index]});
        }
        // 当前页尚未超过 high 时，通过叶链进入下一页。
        current = page.next_page_id;
    }
    return result;
}

std::size_t LocalBPlusIndex::page_count() const
{
    return pages_.size();
}

}  // namespace dbpt


### 4.2 全局目录、节点路由与动态维护

`DistributedBPlusTreeSimulator`管理全局参考记录、物理节点映射和有序目录。它负责初始范围分片、点查询和范围查询，并在节点加入、删除时同步更新节点记录、范围边界、局部索引和目录项。


In [ ]:
%%writefile src/03.03_extra_distributed_bplus_tree/src/simulator.cpp
#include "simulator.hpp"

#include "experiment.hpp"

#include <algorithm>
#include <stdexcept>

namespace dbpt {

bool DirectoryEntry::contains(std::int64_t key) const
{
    // 目录区间采用左右端点都包含的闭区间语义。
    return start_key <= key && key <= end_key;
}

bool DirectoryEntry::overlaps(std::int64_t low, std::int64_t high) const
{
    // 两个闭区间不相交，当且仅当其中一个完全位于另一个左侧或右侧。
    return !(end_key < low || start_key > high);
}

std::size_t DistributedNode::load() const
{
    return records.size();
}

DistributedBPlusTreeSimulator::DistributedBPlusTreeSimulator(std::vector<Record> records,
                                                             std::size_t node_count,
                                                             std::size_t leaf_capacity)
    : records_(std::move(records)), leaf_capacity_(leaf_capacity)
{
    if (records_.empty()) {
        throw std::invalid_argument("records must not be empty");
    }
    if (node_count == 0 || node_count > records_.size()) {
        throw std::invalid_argument("node_count must be in [1, records.size()]");
    }
    // 全局记录按键排序后才能切成互不重叠的连续范围分片。
    std::sort(records_.begin(), records_.end(), [](const Record& lhs, const Record& rhs) {
        return lhs.key < rhs.key;
    });
    build_initial_tree(node_count);
}

const std::vector<DirectoryEntry>& DistributedBPlusTreeSimulator::directory() const
{
    return directory_;
}

const DistributedNode& DistributedBPlusTreeSimulator::node(const std::string& node_id) const
{
    return nodes_.at(node_id);
}

LocalBPlusIndex DistributedBPlusTreeSimulator::build_index(const std::vector<Record>& records) const
{
    // 每个物理节点独立维护一棵局部索引。
    LocalBPlusIndex index(leaf_capacity_);
    index.bulk_load(records);
    return index;
}

DistributedNode DistributedBPlusTreeSimulator::build_node(const std::string& node_id,
                                                          std::int64_t start_key,
                                                          std::int64_t end_key,
                                                          const std::vector<Record>& records) const
{
    // 节点的记录副本保持有序，并据此重新批量构建局部 B+ 树。
    std::vector<Record> node_records = records;
    std::sort(node_records.begin(), node_records.end(), [](const Record& lhs, const Record& rhs) {
        return lhs.key < rhs.key;
    });
    return DistributedNode{node_id, start_key, end_key, node_records, build_index(node_records)};
}

void DistributedBPlusTreeSimulator::build_initial_tree(std::size_t node_count)
{
    // 初始状态按记录数量均分，每个分片对应一个连续键区间和一个物理节点。
    std::vector<std::vector<Record>> chunks = split_evenly(records_, node_count);
    nodes_.clear();
    directory_.clear();

    for (std::size_t index = 0; index < chunks.size(); ++index) {
        const std::string node_id = "node_" + std::to_string(index);
        const std::int64_t start_key = chunks[index].front().key;
        const std::int64_t end_key = chunks[index].back().key;
        DistributedNode node = build_node(node_id, start_key, end_key, chunks[index]);
        nodes_[node_id] = node;
        directory_.push_back(DirectoryEntry{start_key, end_key, node_id});
    }
    sort_directory();
}

void DistributedBPlusTreeSimulator::sort_directory()
{
    // 全局目录按 start_key 排序，范围查询可顺序剪枝，点查询可二分定位。
    std::sort(directory_.begin(), directory_.end(), [](const DirectoryEntry& lhs, const DirectoryEntry& rhs) {
        return lhs.start_key < rhs.start_key;
    });
}

std::vector<std::int64_t> DistributedBPlusTreeSimulator::end_keys() const
{
    std::vector<std::int64_t> keys;
    keys.reserve(directory_.size());
    for (const DirectoryEntry& entry : directory_) {
        keys.push_back(entry.end_key);
    }
    return keys;
}

std::optional<DirectoryEntry> DistributedBPlusTreeSimulator::locate_entry(std::int64_t key) const
{
    if (directory_.empty()) {
        return std::nullopt;
    }
    // 找到第一个 end_key >= key 的候选区间，再用 contains 排除目录间隙。
    const std::vector<std::int64_t> keys = end_keys();
    const auto it = std::lower_bound(keys.begin(), keys.end(), key);
    if (it == keys.end()) {
        return std::nullopt;
    }
    const std::size_t index = static_cast<std::size_t>(std::distance(keys.begin(), it));
    const DirectoryEntry& entry = directory_[index];
    if (!entry.contains(key)) {
        return std::nullopt;
    }
    return entry;
}

std::tuple<std::optional<std::string>, std::optional<std::string>, std::size_t>
DistributedBPlusTreeSimulator::search(std::int64_t key) const
{
    // 第一层访问全局目录，第二层进入目标节点的局部 B+ 树。
    const std::optional<DirectoryEntry> entry = locate_entry(key);
    if (!entry.has_value()) {
        return {std::nullopt, std::nullopt, 0};
    }
    const auto [value, leaf_pages] = nodes_.at(entry->node_id).local_index.search(key);
    return {value, entry->node_id, leaf_pages}; // 查询到的值，路由到的节点，访问的叶页数
}

DistributedRangeResult DistributedBPlusTreeSimulator::range_search(std::int64_t low, std::int64_t high) const
{
    DistributedRangeResult result;
    // 目录有序：跳过查询左侧区间，遇到完全位于查询右侧的区间即可停止。
    for (const DirectoryEntry& entry : directory_) {
        if (entry.end_key < low) {
            continue;
        }
        if (entry.start_key > high) {
            break;
        }
        if (!entry.overlaps(low, high)) {
            continue;
        }
        ++result.touched_nodes;
        // 只向与查询区间相交的节点发起局部范围查询。
        RangeSearchResult local_result = nodes_.at(entry.node_id).local_index.range_search(low, high);
        result.leaf_pages_touched += local_result.leaf_pages_touched;
        result.records.insert(result.records.end(), local_result.records.begin(), local_result.records.end());
    }

    // 合并多个节点的局部结果后，恢复全局键顺序。
    std::sort(result.records.begin(), result.records.end(), [](const Record& lhs, const Record& rhs) {
        return lhs.key < rhs.key;
    });
    return result;
}

MigrationResult DistributedBPlusTreeSimulator::add_node(const std::string& new_node_id)
{
    if (nodes_.count(new_node_id) != 0) {
        throw std::invalid_argument("new node already exists: " + new_node_id);
    }

    // 选择当前记录数最多的节点作为分裂源；负载相同时优先较小键区间。
    auto source_it = std::max_element(nodes_.begin(), nodes_.end(), [](const auto& lhs, const auto& rhs) {
        const DistributedNode& left = lhs.second;
        const DistributedNode& right = rhs.second;
        if (left.load() != right.load()) {
            return left.load() < right.load();
        }
        return left.start_key > right.start_key;
    });

    DistributedNode source = source_it->second;
    if (source.load() < 2) {
        throw std::runtime_error("source node has too few records to split");
    }

    // 按有序记录中点切分，只迁移源节点后一半连续记录。
    const std::size_t split_index = source.load() / 2;
    std::vector<Record> left_records(source.records.begin(), source.records.begin() + static_cast<std::ptrdiff_t>(split_index));
    std::vector<Record> right_records(source.records.begin() + static_cast<std::ptrdiff_t>(split_index), source.records.end());
    const std::int64_t split_key = right_records.front().key;

    // 分裂后为源节点和新节点分别重建范围及局部索引。
    DistributedNode updated_source = build_node(source.node_id, source.start_key, split_key - 1, left_records);
    DistributedNode new_node = build_node(new_node_id, split_key, source.end_key, right_records);

    nodes_[updated_source.node_id] = updated_source;
    nodes_[new_node_id] = new_node;

    // 用两个新目录项替换原源节点目录项，然后恢复目录顺序。
    directory_.erase(std::remove_if(directory_.begin(), directory_.end(), [&](const DirectoryEntry& entry) {
                         return entry.node_id == source.node_id;
                     }),
                     directory_.end());
    directory_.push_back(DirectoryEntry{updated_source.start_key, updated_source.end_key, updated_source.node_id});
    directory_.push_back(DirectoryEntry{new_node.start_key, new_node.end_key, new_node.node_id});
    sort_directory();

    const std::size_t migrated = right_records.size();
    return MigrationResult{"add_node", source.node_id, new_node_id, migrated, static_cast<double>(migrated) / records_.size()};
}

MigrationResult DistributedBPlusTreeSimulator::remove_node(const std::string& node_id)
{
    if (nodes_.count(node_id) == 0) {
        throw std::invalid_argument("node does not exist: " + node_id);
    }
    if (directory_.size() <= 1) {
        throw std::runtime_error("at least one node must remain");
    }

    const auto entry_it = std::find_if(directory_.begin(), directory_.end(), [&](const DirectoryEntry& entry) {
        return entry.node_id == node_id;
    });
    const std::size_t index = static_cast<std::size_t>(std::distance(directory_.begin(), entry_it));

    // 为保持键区间连续，只允许迁移到被删除节点的左邻居或右邻居。
    std::vector<DistributedNode> candidates;
    if (index > 0) {
        candidates.push_back(nodes_.at(directory_[index - 1].node_id));
    }
    if (index + 1 < directory_.size()) {
        candidates.push_back(nodes_.at(directory_[index + 1].node_id));
    }

    // 在相邻候选中选择负载较小者，负载相同时选择起始键较小者。
    const DistributedNode& target = *std::min_element(candidates.begin(), candidates.end(), [](const DistributedNode& lhs,
                                                                                               const DistributedNode& rhs) {
        if (lhs.load() != rhs.load()) {
            return lhs.load() < rhs.load();
        }
        return lhs.start_key < rhs.start_key;
    });
    const DistributedNode leaving = nodes_.at(node_id);

    // 合并离开节点和目标节点的记录，并重建一个覆盖两段范围的局部索引。
    std::vector<Record> merged_records = target.records;
    merged_records.insert(merged_records.end(), leaving.records.begin(), leaving.records.end());
    std::sort(merged_records.begin(), merged_records.end(), [](const Record& lhs, const Record& rhs) {
        return lhs.key < rhs.key;
    });

    const std::int64_t merged_start = std::min(target.start_key, leaving.start_key);
    const std::int64_t merged_end = std::max(target.end_key, leaving.end_key);
    DistributedNode updated_target = build_node(target.node_id, merged_start, merged_end, merged_records);

    nodes_.erase(node_id);
    nodes_[updated_target.node_id] = updated_target;

    // 删除旧的两个目录项，以合并后的目标节点目录项替换。
    directory_.erase(std::remove_if(directory_.begin(), directory_.end(), [&](const DirectoryEntry& entry) {
                         return entry.node_id == node_id || entry.node_id == target.node_id;
                     }),
                     directory_.end());
    directory_.push_back(DirectoryEntry{updated_target.start_key, updated_target.end_key, updated_target.node_id});
    sort_directory();

    const std::size_t migrated = leaving.records.size();
    return MigrationResult{"remove_node", node_id, target.node_id, migrated, static_cast<double>(migrated) / records_.size()};
}

std::map<std::string, std::size_t> DistributedBPlusTreeSimulator::node_loads() const
{
    std::map<std::string, std::size_t> loads;
    for (const DirectoryEntry& entry : directory_) {
        loads[entry.node_id] = nodes_.at(entry.node_id).load();
    }
    return loads;
}

std::map<std::string, std::size_t> DistributedBPlusTreeSimulator::leaf_pages_per_node() const
{
    std::map<std::string, std::size_t> pages;
    for (const DirectoryEntry& entry : directory_) {
        pages[entry.node_id] = nodes_.at(entry.node_id).local_index.page_count();
    }
    return pages;
}

std::map<std::int64_t, std::string> DistributedBPlusTreeSimulator::assignment() const
{
    // 展开当前分布状态为 key -> node_id，便于统计迁移和校验全局键集合。
    std::map<std::int64_t, std::string> result;
    for (const DirectoryEntry& entry : directory_) {
        for (const Record& record : nodes_.at(entry.node_id).records) {
            result[record.key] = entry.node_id;
        }
    }
    return result;
}

std::pair<bool, std::string> DistributedBPlusTreeSimulator::validate_directory() const
{
    if (directory_.empty()) {
        return {false, "directory is empty"};
    }
    // 相邻目录项必须严格不重叠；允许键空间中存在没有记录的间隙。
    for (std::size_t index = 1; index < directory_.size(); ++index) {
        if (directory_[index - 1].end_key >= directory_[index].start_key) {
            return {false, "directory ranges overlap"};
        }
    }
    return {true, "PASS"};
}

std::pair<bool, std::string> DistributedBPlusTreeSimulator::validate_records() const
{
    // 先验证迁移前后的全局键集合完全一致，防止丢失或重复记录。
    std::vector<std::int64_t> actual_keys;
    actual_keys.reserve(records_.size());
    for (const auto& item : assignment()) {
        actual_keys.push_back(item.first);
    }

    std::vector<std::int64_t> expected_keys;
    expected_keys.reserve(records_.size());
    for (const Record& record : records_) {
        expected_keys.push_back(record.key);
    }

    if (actual_keys != expected_keys) {
        return {false, "global key set mismatch"};
    }

    // 再验证每条记录都落在其节点目录项声明的范围内。
    for (const DirectoryEntry& entry : directory_) {
        const DistributedNode& current = nodes_.at(entry.node_id);
        for (const Record& record : current.records) {
            if (!entry.contains(record.key)) {
                return {false, "record is outside its directory range"};
            }
        }
    }
    return {true, "PASS"};
}

}  // namespace dbpt


### 4.3 负载摘要与全局重分布基线

`compute_load_stats`根据节点记录数计算最小值、最大值、平均值、总体标准差和不均衡率。`global_redistribution_assignment`模拟节点数量变化后对全部记录重新均分，`count_assignment_changes`统计与原分配相比发生归属变化的记录数。


In [ ]:
%%writefile src/03.03_extra_distributed_bplus_tree/src/metrics.cpp
#include "metrics.hpp"

#include "experiment.hpp"

#include <algorithm>
#include <cmath>
#include <numeric>

namespace dbpt {

LoadStats compute_load_stats(const std::string& label, const std::map<std::string, std::size_t>& loads)
{
    // 将各节点记录数转为数值序列，用于计算总体均值、方差和不均衡度。
    std::vector<double> values;
    values.reserve(loads.size());
    for (const auto& item : loads) {
        values.push_back(static_cast<double>(item.second));
    }

    const auto [min_it, max_it] = std::minmax_element(values.begin(), values.end());
    const double total = std::accumulate(values.begin(), values.end(), 0.0);
    const double avg = values.empty() ? 0.0 : total / values.size();
    double variance = 0.0;
    for (double value : values) {
        const double diff = value - avg;
        variance += diff * diff;
    }
    variance = values.empty() ? 0.0 : variance / values.size();

    LoadStats stats;
    stats.label = label;
    stats.node_loads = loads;
    stats.min_load = values.empty() ? 0 : static_cast<std::size_t>(*min_it);
    stats.max_load = values.empty() ? 0 : static_cast<std::size_t>(*max_it);
    stats.avg_load = avg;
    stats.std_load = std::sqrt(variance);
    // 最大负载与平均负载越接近，imbalance_ratio 越接近 1。
    stats.imbalance_ratio = avg == 0.0 ? 0.0 : static_cast<double>(stats.max_load) / avg;
    return stats;
}

std::map<std::int64_t, std::string> global_redistribution_assignment(const std::vector<Record>& records,
                                                                     std::size_t node_count)
{
    // 对照方案会在节点数变化后重新均分全部记录，可能引发大范围迁移。
    std::vector<Record> sorted_records = records;
    std::sort(sorted_records.begin(), sorted_records.end(), [](const Record& lhs, const Record& rhs) {
        return lhs.key < rhs.key;
    });

    std::map<std::int64_t, std::string> assignment;
    const std::vector<std::vector<Record>> chunks = split_evenly(sorted_records, node_count);
    for (std::size_t node_index = 0; node_index < chunks.size(); ++node_index) {
        const std::string node_id = "node_" + std::to_string(node_index);
        for (const Record& record : chunks[node_index]) {
            assignment[record.key] = node_id;
        }
    }
    return assignment;
}

std::size_t count_assignment_changes(const std::map<std::int64_t, std::string>& before,
                                     const std::map<std::int64_t, std::string>& after)
{
    // 同一键在前后映射中的节点不同，即视为发生一次迁移。
    std::size_t changes = 0;
    for (const auto& item : before) {
        const auto after_it = after.find(item.first);
        if (after_it == after.end() || after_it->second != item.second) {
            ++changes;
        }
    }
    return changes;
}

}  // namespace dbpt


### 4.4 实验场景、查询验证与结果输出

`run_experiment`按“生成记录→构建初始索引→查询验证→节点加入→计算重分布基线→节点删除→再次验证”的顺序组织实验。串行`map`和全量扫描结果作为独立参考，用于检查点查询和范围查询正确性。


In [ ]:
%%writefile src/03.03_extra_distributed_bplus_tree/src/experiment.cpp
#include "experiment.hpp"

#include <algorithm>
#include <iomanip>
#include <iostream>
#include <random>
#include <set>
#include <sstream>
#include <stdexcept>

namespace dbpt {

namespace {

std::string value_for_key(std::int64_t key)
{
    // 根据键生成确定性 value，便于查询结果与串行参考结果逐项比较。
    std::ostringstream oss;
    oss << "value_" << std::setw(6) << std::setfill('0') << key;
    return oss.str();
}

std::size_t safe_index(std::size_t requested, std::size_t size)
{
    if (size == 0) {
        return 0;
    }
    return std::min(requested, size - 1);
}

void print_config(const ExperimentConfig& config)
{
    // 输出函数只负责展示，不参与索引构建和迁移逻辑。
    std::cout << "Experiment configuration\n";
    std::cout << std::string(88, '-') << "\n";
    std::cout << "record_count        : " << config.record_count << "\n";
    std::cout << "initial_nodes       : " << config.initial_nodes << "\n";
    std::cout << "leaf_capacity       : " << config.leaf_capacity << "\n";
    std::cout << "key_distribution    : " << config.key_distribution << "\n";
    std::cout << "add_node_id         : " << config.add_node_id << "\n";
    std::cout << "remove_node_id      : " << config.remove_node_id << "\n";
    std::cout << "seed                : " << config.seed << "\n";
    std::cout << std::string(88, '-') << "\n";
}

void print_directory(const DistributedBPlusTreeSimulator& simulator, const std::string& title)
{
    std::cout << "\n" << title << "\n";
    std::cout << std::string(72, '-') << "\n";
    std::cout << std::left << std::setw(10) << "node" << std::right << std::setw(12) << "start_key"
              << std::setw(12) << "end_key" << std::setw(10) << "records" << std::setw(12) << "leaf_pages"
              << "\n";
    std::cout << std::string(72, '-') << "\n";
    const auto pages = simulator.leaf_pages_per_node();
    for (const DirectoryEntry& entry : simulator.directory()) {
        const DistributedNode& node = simulator.node(entry.node_id);
        std::cout << std::left << std::setw(10) << entry.node_id << std::right << std::setw(12) << entry.start_key
                  << std::setw(12) << entry.end_key << std::setw(10) << node.load() << std::setw(12)
                  << pages.at(entry.node_id) << "\n";
    }
    std::cout << std::string(72, '-') << "\n";
}

void print_load_stats(const std::vector<LoadStats>& stats_list)
{
    std::cout << "\nLoad statistics\n";
    std::cout << std::string(88, '-') << "\n";
    std::cout << std::left << std::setw(18) << "label" << std::right << std::setw(8) << "min" << std::setw(8)
              << "max" << std::setw(12) << "avg" << std::setw(12) << "std" << std::setw(12) << "imbalance"
              << "\n";
    std::cout << std::string(88, '-') << "\n";
    for (const LoadStats& stats : stats_list) {
        std::cout << std::left << std::setw(18) << stats.label << std::right << std::setw(8) << stats.min_load
                  << std::setw(8) << stats.max_load << std::setw(12) << std::fixed << std::setprecision(2)
                  << stats.avg_load << std::setw(12) << stats.std_load << std::setw(12) << std::setprecision(3)
                  << stats.imbalance_ratio << "\n";
    }
    std::cout << std::string(88, '-') << "\n";
}

void print_query_results(const std::vector<QueryResult>& results)
{
    std::cout << "\nQuery correctness\n";
    std::cout << std::string(100, '-') << "\n";
    std::cout << std::left << std::setw(32) << "query" << std::right << std::setw(10) << "correct"
              << std::setw(14) << "result_count" << std::setw(8) << "nodes" << std::setw(12) << "leaf_pages"
              << "\n";
    std::cout << std::string(100, '-') << "\n";
    for (const QueryResult& result : results) {
        std::cout << std::left << std::setw(32) << result.query << std::right << std::setw(10)
                  << (result.correct ? "true" : "false") << std::setw(14) << result.result_count << std::setw(8)
                  << result.touched_nodes << std::setw(12) << result.leaf_pages_touched << "\n";
    }
    std::cout << std::string(100, '-') << "\n";
}

void print_migration_results(const std::vector<MigrationResult>& results,
                             std::size_t full_redistribution_add,
                             std::size_t total)
{
    std::cout << "\nMigration summary\n";
    std::cout << std::string(108, '-') << "\n";
    std::cout << std::left << std::setw(26) << "event" << std::setw(10) << "source" << std::setw(10) << "target"
              << std::right << std::setw(12) << "migrated" << std::setw(12) << "ratio" << "\n";
    std::cout << std::string(108, '-') << "\n";
    for (const MigrationResult& result : results) {
        std::ostringstream ratio;
        ratio << std::fixed << std::setprecision(2) << result.migration_ratio * 100.0 << "%";
        std::cout << std::left << std::setw(26) << result.event << std::setw(10) << result.source_node
                  << std::setw(10) << result.target_node << std::right << std::setw(12) << result.migrated_records
                  << std::setw(12) << ratio.str() << "\n";
    }
    std::ostringstream ratio;
    ratio << std::fixed << std::setprecision(2) << static_cast<double>(full_redistribution_add) * 100.0 / total << "%";
    std::cout << std::left << std::setw(26) << "global_redistribute_add" << std::setw(10) << "all" << std::setw(10)
              << "all" << std::right << std::setw(12) << full_redistribution_add << std::setw(12) << ratio.str()
              << "\n";
    std::cout << std::string(108, '-') << "\n";
}

}  // namespace

std::vector<Record> generate_records(const std::string& mode, std::size_t record_count, unsigned seed)
{
    if (record_count == 0) {
        throw std::invalid_argument("record_count must be positive");
    }

    std::vector<std::int64_t> keys;
    std::mt19937 rng(seed);

    if (mode == "uniform") {
        // 连续递增键，形成最规则的均匀范围分片。
        keys.reserve(record_count);
        for (std::size_t index = 0; index < record_count; ++index) {
            keys.push_back(static_cast<std::int64_t>(index));
        }
    } else if (mode == "clustered") {
        // 在四个相距较远的区域生成键，模拟存在明显空洞的簇状键空间。
        std::set<std::int64_t> unique_keys;
        const std::size_t cluster_count = 4;
        const std::size_t cluster_size = (record_count + cluster_count - 1) / cluster_count;
        std::uniform_int_distribution<int> bit_dist(0, 1);
        for (std::size_t cluster_id = 0; cluster_id < cluster_count; ++cluster_id) {
            const std::int64_t base = static_cast<std::int64_t>(cluster_id * record_count * 3);
            for (std::size_t offset = 0; offset < cluster_size; ++offset) {
                unique_keys.insert(base + static_cast<std::int64_t>(offset * 2) + bit_dist(rng));
            }
        }
        keys.assign(unique_keys.begin(), unique_keys.end());
        if (keys.size() > record_count) {
            keys.resize(record_count);
        }
        std::uniform_int_distribution<int> step_dist(1, 3);
        while (keys.size() < record_count) {
            keys.push_back(keys.back() + step_dist(rng));
        }
    } else if (mode == "skewed") {
        // 70% 键集中在低位连续区间，其余键稀疏分布在远端冷区间。
        const std::size_t hot_count = static_cast<std::size_t>(record_count * 0.7);
        const std::size_t cold_count = record_count - hot_count;
        keys.reserve(record_count);
        for (std::size_t index = 0; index < hot_count; ++index) {
            keys.push_back(static_cast<std::int64_t>(index));
        }
        for (std::size_t index = 0; index < cold_count; ++index) {
            keys.push_back(static_cast<std::int64_t>(record_count * 5 + index * 10));
        }
        std::sort(keys.begin(), keys.end());
    } else {
        throw std::invalid_argument("unsupported key_distribution: " + mode);
    }

    // 将生成的唯一键转换为可查询的键值记录。
    std::vector<Record> records;
    records.reserve(record_count);
    for (std::int64_t key : keys) {
        records.push_back(Record{key, value_for_key(key)});
        if (records.size() == record_count) {
            break;
        }
    }
    return records;
}

std::vector<std::vector<Record>> split_evenly(const std::vector<Record>& records, std::size_t parts)
{
    if (parts == 0 || parts > records.size()) {
        throw std::invalid_argument("parts must be in [1, records.size()]");
    }

    std::vector<std::vector<Record>> result;
    result.reserve(parts);
    const std::size_t total = records.size();
    // 使用整数比例边界切分，分片大小最多相差一条记录且保持键连续。
    for (std::size_t index = 0; index < parts; ++index) {
        const std::size_t start = index * total / parts;
        const std::size_t end = (index + 1) * total / parts;
        result.emplace_back(records.begin() + static_cast<std::ptrdiff_t>(start),
                            records.begin() + static_cast<std::ptrdiff_t>(end));
    }
    return result;
}

std::vector<QueryResult> validate_queries(const DistributedBPlusTreeSimulator& simulator,
                                          const std::vector<Record>& records)
{
    // map 作为与分布式目录及局部索引无关的串行正确性参考。
    std::map<std::int64_t, std::string> reference;
    for (const Record& record : records) {
        reference[record.key] = record.value;
    }

    std::vector<QueryResult> results;
    // 点查询覆盖首、中、尾部存在键以及一个明确不存在的键。
    const std::vector<std::int64_t> keys = {
        records.front().key,
        records[safe_index(records.size() / 3, records.size())].key,
        records[safe_index(records.size() / 2, records.size())].key,
        records.back().key,
        records.back().key + 999999,
    };

    for (std::int64_t key : keys) {
        const auto [value, node_id, leaf_pages] = simulator.search(key);
        const auto ref_it = reference.find(key);
        const bool correct = value.has_value() == (ref_it != reference.end()) &&
                             (!value.has_value() || value.value() == ref_it->second);
        results.push_back(QueryResult{
            "search(" + std::to_string(key) + ")",
            correct,
            value.has_value() ? 1u : 0u,
            node_id.has_value() ? 1u : 0u,
            leaf_pages,
        });
    }

    // 范围查询通过记录下标选取边界，可兼容不同键分布模式。
    const std::vector<std::pair<std::size_t, std::size_t>> query_indices = {
        {100, 160},
        {2400, 2700},
        {4900, 5300},
        {8000, 9500},
    };

    for (const auto& pair : query_indices) {
        const std::int64_t low = records[safe_index(pair.first, records.size())].key;
        const std::int64_t high = records[safe_index(pair.second, records.size())].key;
        DistributedRangeResult got = simulator.range_search(low, high);

        // 直接扫描全量记录生成期望结果，再与分布式范围查询逐条比较。
        std::vector<Record> expected;
        for (const Record& record : records) {
            if (low <= record.key && record.key <= high) {
                expected.push_back(record);
            }
        }

        const bool correct = got.records.size() == expected.size() &&
                             std::equal(got.records.begin(), got.records.end(), expected.begin(), [](const Record& lhs,
                                                                                                      const Record& rhs) {
                                 return lhs.key == rhs.key && lhs.value == rhs.value;
                             });

        results.push_back(QueryResult{
            "range_search(" + std::to_string(low) + ", " + std::to_string(high) + ")",
            correct,
            got.records.size(),
            got.touched_nodes,
            got.leaf_pages_touched,
        });
    }

    return results;
}

void run_experiment(const ExperimentConfig& config)
{
    // 步骤一：构建初始分区、全局目录和本地索引。
    const std::vector<Record> records = generate_records(config.key_distribution, config.record_count, config.seed);
    DistributedBPlusTreeSimulator simulator(records, config.initial_nodes, config.leaf_capacity);

    print_config(config);
    print_directory(simulator, "Initial directory");

    // 步骤二：执行点查询和范围查询
    const auto before_add_assignment = simulator.assignment();
    const std::vector<QueryResult> query_results_before = validate_queries(simulator, records);
    const LoadStats initial_stats = compute_load_stats("initial", simulator.node_loads());

    // 步骤三：模拟结点加入与局部分区拆分
    const MigrationResult add_result = simulator.add_node(config.add_node_id);
    print_directory(simulator, "Directory after node add");
    const LoadStats after_add_stats = compute_load_stats("after_add", simulator.node_loads());

    // 计算新增节点时全量重新均分的迁移量，作为局部迁移的对照基线。
    const auto global_add_assignment = global_redistribution_assignment(records, config.initial_nodes + 1);
    const std::size_t full_redistribution_add = count_assignment_changes(before_add_assignment, global_add_assignment);

    // 步骤四：模拟结点移除与相邻分区合并。
    const MigrationResult remove_result = simulator.remove_node(config.remove_node_id);
    print_directory(simulator, "Directory after node remove");
    const LoadStats after_remove_stats = compute_load_stats("after_remove", simulator.node_loads());

    // 验证目录不重叠、记录未丢失且迁移后查询结果仍正确。
    const auto [directory_ok, directory_message] = simulator.validate_directory();
    const auto [records_ok, records_message] = simulator.validate_records();
    const std::vector<QueryResult> query_results_after = validate_queries(simulator, records);

    std::vector<QueryResult> all_queries = query_results_before;
    all_queries.insert(all_queries.end(), query_results_after.begin(), query_results_after.end());

    // 步骤五：结果校验和指标对比。
    print_load_stats({initial_stats, after_add_stats, after_remove_stats});
    print_query_results(all_queries);
    print_migration_results({add_result, remove_result}, full_redistribution_add, records.size());

    const bool query_ok = std::all_of(all_queries.begin(), all_queries.end(), [](const QueryResult& result) {
        return result.correct;
    });

    std::cout << "\nCorrectness check\n";
    std::cout << std::string(88, '-') << "\n";
    std::cout << "directory            : " << directory_message << "\n";
    std::cout << "records              : " << records_message << "\n";
    std::cout << "query_result         : " << (query_ok ? "PASS" : "FAIL") << "\n";
    std::cout << "directory_ok         : " << (directory_ok ? "true" : "false") << "\n";
    std::cout << "records_ok           : " << (records_ok ? "true" : "false") << "\n";
    std::cout << "query_count          : " << all_queries.size() << "\n";
    std::cout << std::string(88, '-') << "\n";
}

}  // namespace dbpt


### 4.5 主程序与命令行参数

主程序负责解析记录数、初始节点数、叶页容量、键分布、增删节点ID和随机种子等参数，再调用实验流程。`--key-distribution`支持`uniform`、`clustered`和`skewed`三种模式。


In [ ]:
%%writefile src/03.03_extra_distributed_bplus_tree/src/main.cpp
#include "experiment.hpp"

#include <cstdlib>
#include <iostream>
#include <stdexcept>
#include <string>

namespace {

void print_help(const char* program)
{
    std::cout << "Usage: " << program << " [options]\n\n";
    std::cout << "Options:\n";
    std::cout << "  --record-count N          Number of records, default 10000\n";
    std::cout << "  --initial-nodes N         Initial node count, default 4\n";
    std::cout << "  --leaf-capacity N         Leaf page capacity, default 32\n";
    std::cout << "  --key-distribution MODE   uniform, clustered, or skewed\n";
    std::cout << "  --add-node-id ID          New node id, default node_4\n";
    std::cout << "  --remove-node-id ID       Removed node id, default node_1\n";
    std::cout << "  --seed N                  Random seed, default 2026\n";
    std::cout << "  --no-draw                 Accepted for compatibility with the Python version\n";
    std::cout << "  --help                    Show this help message\n";
}

std::string require_value(int argc, char** argv, int& index)
{
    // 统一读取命令行选项后的参数，并在缺少参数值时立即报错。
    if (index + 1 >= argc) {
        throw std::invalid_argument(std::string("missing value for ") + argv[index]);
    }
    ++index;
    return argv[index];
}

dbpt::ExperimentConfig parse_args(int argc, char** argv)
{
    // 从默认配置开始，再用命令行选项逐项覆盖。
    dbpt::ExperimentConfig config;
    for (int index = 1; index < argc; ++index) {
        const std::string arg = argv[index];
        if (arg == "--help" || arg == "-h") {
            print_help(argv[0]);
            std::exit(0);
        } else if (arg == "--record-count") {
            config.record_count = static_cast<std::size_t>(std::stoull(require_value(argc, argv, index)));
        } else if (arg == "--initial-nodes") {
            config.initial_nodes = static_cast<std::size_t>(std::stoull(require_value(argc, argv, index)));
        } else if (arg == "--leaf-capacity") {
            config.leaf_capacity = static_cast<std::size_t>(std::stoull(require_value(argc, argv, index)));
        } else if (arg == "--key-distribution") {
            config.key_distribution = require_value(argc, argv, index);
        } else if (arg == "--add-node-id") {
            config.add_node_id = require_value(argc, argv, index);
        } else if (arg == "--remove-node-id") {
            config.remove_node_id = require_value(argc, argv, index);
        } else if (arg == "--seed") {
            config.seed = static_cast<unsigned>(std::stoul(require_value(argc, argv, index)));
        } else if (arg == "--no-draw") {
            continue;
        } else {
            throw std::invalid_argument("unknown argument: " + arg);
        }
    }
    return config;
}

}  // namespace

int main(int argc, char** argv)
{
    try {
        // main 只负责参数解析和实验调度，核心流程位于 run_experiment。
        const dbpt::ExperimentConfig config = parse_args(argc, argv);
        dbpt::run_experiment(config);
    } catch (const std::exception& error) {
        std::cerr << "error: " << error.what() << "\n";
        return 1;
    }
    return 0;
}


### 4.6 核心源码检查

检查全部头文件和源文件是否已经写入。全部显示`OK`时，说明Notebook生成的C++工程源码完整，可以进入构建和运行阶段。


In [ ]:
required_files = [
    "include/record.hpp",
    "include/directory.hpp",
    "include/local_bplus_index.hpp",
    "include/simulator.hpp",
    "include/metrics.hpp",
    "include/experiment.hpp",
    "src/local_bplus_index.cpp",
    "src/simulator.cpp",
    "src/metrics.cpp",
    "src/experiment.cpp",
    "src/main.cpp",
]

for relative_path in required_files:
    path = WORK_DIR / relative_path
    print(f"{relative_path}: {'OK' if path.exists() else 'MISSING'}")


---
## 5. 结果验证与性能分析

核心源码准备完成后，本节写入CMake构建配置和运行脚本，运行默认实验并保存输出。由于本实验是单机模拟，不需要NPU或多进程启动，运行结果可以直接在Notebook中复现。


### 5.1 工程构建

`CMakeLists.txt`使用C++17组织5个源文件，启用`-Wall -Wextra -pedantic`警告选项。工程只依赖C++标准库，适合在普通Linux服务器或Notebook环境中构建。


In [ ]:
%%writefile src/03.03_extra_distributed_bplus_tree/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

project(distributed_bplus_tree_experiment LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)

add_executable(distributed_bplus_tree
    src/main.cpp
    src/local_bplus_index.cpp
    src/simulator.cpp
    src/metrics.cpp
    src/experiment.cpp
)

target_include_directories(distributed_bplus_tree PRIVATE include)

if (MSVC)
    target_compile_options(distributed_bplus_tree PRIVATE /W4 /utf-8)
else()
    target_compile_options(distributed_bplus_tree PRIVATE -Wall -Wextra -pedantic)
endif()


### 5.2 编译和运行脚本

`scripts/run.sh`在工程根目录执行CMake配置、编译和可执行文件运行三个步骤，并把Notebook传入的命令行参数继续传递给程序。脚本采用`set -euo pipefail`，任一构建或运行步骤失败都会返回非零状态。


In [ ]:
%%writefile src/03.03_extra_distributed_bplus_tree/scripts/run.sh
#!/usr/bin/env bash
set -euo pipefail

PROJECT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")/.." && pwd)"

cmake -S "${PROJECT_DIR}" -B "${PROJECT_DIR}/build"
cmake --build "${PROJECT_DIR}/build"
"${PROJECT_DIR}/build/distributed_bplus_tree" "$@"


In [ ]:
!chmod +x src/03.03_extra_distributed_bplus_tree/scripts/run.sh
!find src/03.03_extra_distributed_bplus_tree -maxdepth 3 -type f | sort


### 5.3 运行默认实验

下面的命令使用默认配置完成编译和运行：10000条记录、4个初始节点、叶页容量32和均匀键分布。`tee`会把终端输出同时保存到`results/notebook_result.txt`，便于后续读取。


In [ ]:
!cd src/03.03_extra_distributed_bplus_tree && bash scripts/run.sh | tee results/notebook_result.txt


### 5.4 运行参数对照实验

完成默认实验后，可以改变记录数、叶页容量和键分布，观察范围边界、叶页访问数量和迁移指标的变化。下面使用2000条记录、叶页容量16和倾斜键分布执行一组较小规模实验。


In [ ]:
!cd src/03.03_extra_distributed_bplus_tree && bash scripts/run.sh \
  --record-count 2000 \
  --initial-nodes 4 \
  --leaf-capacity 16 \
  --key-distribution skewed \
  --add-node-id node_4 \
  --remove-node-id node_1 \
  --seed 2026 \
  | tee results/skewed_notebook_result.txt


### 5.5 读取实验结果

下面读取保存的默认实验输出。重点观察`Initial directory`、`Directory after node add`、`Directory after node remove`、`Load statistics`、`Query correctness`、`Migration summary`和`Correctness check`等部分。


In [ ]:
result_path = WORK_DIR / "results" / "notebook_result.txt"
if result_path.exists():
    print(result_path.read_text(encoding="utf-8", errors="ignore")[:12000])
else:
    print("not found:", result_path)


### 5.6 结果验证

结果验证主要从目录、记录和查询三个方面进行：

1. 目录项按起始键升序排列，相邻区间不得重叠，每个目录项引用的节点必须存在；
2. 所有节点保存的记录总数应等于原始记录数，每个键只能出现一次，节点记录必须落在自身范围内；
3. 初始状态和节点增删后的点查询、范围查询都应与串行参考结果一致；
4. 最终输出中`directory_ok`、`records_ok`应为`true`，`query_result`应为`PASS`；
5. 迁移记录数不得超过总记录数，迁移比例应处于$[0,1]$范围。


### 5.7 局部迁移与全局重分布分析

节点加入时，局部策略只从最重节点迁移后一半记录；全局重新均分则可能改变多个原节点的范围边界。默认初始状态下4个节点各有2500条记录，局部拆分约迁移1250条，即总记录的12.5%。全局5节点均分的理论平均负载为2000条，但需要移动多个边界附近的记录，迁移范围通常更大。

因此，局部策略以一定的暂时负载偏差换取更小的迁移范围。报告中应以程序实际输出的`add_node`和`global_redistribute_add`两行作定量比较，并说明记录规模、节点数和被拆分节点都会影响比例。


### 5.8 负载与查询代价分析

初始按记录数均分时，不均衡率应接近1。节点加入后，原最重节点被拆成两个较小分区，而其他节点保持原负载，因此实际负载不会立即达到5节点全局均分状态；节点删除后，被删分区并入较轻相邻节点，局部范围连续性得到保持，但目标节点负载可能上升。

叶页容量越小，同样记录数需要的叶页越多，跨越相同记录范围时通常访问更多叶页。键分布主要改变目录边界和查询键值跨度；在按记录数量切分的前提下，它不一定直接造成初始记录数负载倾斜。应结合`nodes`、`leaf_pages`和`result_count`三列分析查询访问代价。


In [ ]:
import re

output_path = WORK_DIR / "results" / "notebook_result.txt"
if output_path.exists():
    output = output_path.read_text(encoding="utf-8", errors="ignore")
    checks = {
        "directory_ok": re.search(r"^directory_ok\s*:\s*(\w+)", output, re.MULTILINE),
        "records_ok": re.search(r"^records_ok\s*:\s*(\w+)", output, re.MULTILINE),
        "query_result": re.search(r"^query_result\s*:\s*(\w+)", output, re.MULTILINE),
    }
    for name, match in checks.items():
        print(f"{name}: {match.group(1) if match else 'NOT FOUND'}")

    print("\nMigration rows:")
    for line in output.splitlines():
        if line.startswith("add_node") or line.startswith("remove_node") or line.startswith("global_redistribute_add"):
            print(line)
else:
    print("not found:", output_path)


---
## 6. 实验总结

本实验按照实验概述、环境准备、问题分析、核心程序开发和结果验证与性能分析五个阶段，实现了分布式B+树结构动态更新与局部数据迁移的单机模拟。

* 全局目录使用连续闭区间把键路由到物理节点，局部B+树负责节点内记录查询。
* 局部索引通过页高键二分定位叶页，范围查询沿叶页链顺序扫描。
* 点查询通常只访问一个目标节点，范围查询只访问与查询区间相交的节点并合并局部结果。
* 节点加入通过拆分最重节点完成局部迁移，节点删除通过合并到较轻相邻节点保持范围连续性。
* 全局重新均分基线展示了更理想负载可能带来的额外迁移成本，局部策略体现了迁移范围与负载均衡之间的权衡。
* 目录、记录和查询三类正确性检查保证动态更新后没有区间重叠、记录丢失或查询错误。

通过本实验，可以理解分布式范围索引从全局路由、局部查询到动态维护的完整流程，掌握局部数据迁移的组织方法，并具备结合负载、查询访问和迁移指标分析分布式索引维护成本的能力。
